# 002: Understanding the Framework Layer

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import ibis

from earlysign.core.ledger import Ledger

connection = ibis.connect("duckdb://:memory:")
ledger_bare = Ledger(connection, "example_bare").bind(exp_id="exp_001")
ledger_struct = Ledger(connection, "example_structured").bind(exp_id="exp_001")

ledger_bare.ensure()
ledger_struct.ensure()

ledger_bare, ledger_struct

## LedgerRecord

If we do this directly using the ledger syntax:

In [ ]:
ledger_bare.insert(
    data={"nA": 100, "nB": 100, "mA": 10, "mB": 35},
    attributes={"record_id": "example_data_protocol"},
)
ledger_bare.show()

Instead, we can use the framework layer:

In [ ]:
from earlysign.v0.framework.records import LedgerRecord, QueryMixin


class Data(LedgerRecord, QueryMixin):
    schema = {"nA": int, "nB": int, "mA": int, "mB": int}


# Instantiate and attach to the ledger
data_adapter = Data(name="example_data_protocol")
data_adapter.attach(ledger_struct)

# Insert a new record
data_adapter.insert(nA=100, nB=100, mA=10, mB=35)

ledger_struct.show()

Now, suppose we want to read these records. Let's say we have some more data written in the ledger by other operations.

In [ ]:
ledger_bare.insert(data={"some": "other", "data": "by other operations"})
ledger_bare.insert(data={"some": "other", "data": "by other operations"})
ledger_bare.insert(data={"some": "other", "data": "by other operations"})
ledger_bare.show()

In [ ]:
ledger_struct.insert(data={"some": "other", "data": "by other operations"})
ledger_struct.insert(data={"some": "other", "data": "by other operations"})
ledger_struct.insert(data={"some": "other", "data": "by other operations"})
ledger_struct.show()

Then, in order to read out the records using only the bare ledger functionality, you need to carefully write the logic to query the relevant data record based on the label values.

In [ ]:
ledger_bare.t.filter(
    ledger_bare.t.attributes["record_id"].str == "example_data_protocol"
).execute()
# type(ledger_bare.t.select(ledger_bare.t.attributes["record_id"]).execute().to_dict()["JSONGetItem(labels, 'record_id')"][0])
# print('ledger_bare.t.attributes["record_id"].cast("string").execute()')
# print(ledger_bare.t.attributes["record_id"].cast("string").execute())
# print('ledger_bare.t.attributes["record_id"].str.execute()')
# print(ledger_bare.t.attributes["record_id"].str.execute())

On the other hand, if you are using the framework layer, the same can be achieved by simple calls of the helper methods without specifying the query on your own; the LedgerRecord subclass encapsulates the boilerplate and does the work for you.
Compare this code against the above.

In [ ]:
data_adapter.latest().execute()

This adds another layer of abstraction that comes in handy in more complex scenarios,
where complex calculation operations can focus on the logic itself, not on how to read out the appropriate ledger record from the history or other boilerplate.

## LedgerOp

LedgerOps take one or more LedgerRecords at the time of instantiation,
reads the contents of the ledger through the LedgerRecord instances,
and writes new events to the ledger through the LedgerRecord instances.

Conceptually, they represent operations on the ledger.
They perform the read/write actions only through the LedgerRecords.

In [ ]:
from earlysign.v0.framework.operator import LedgerOp


class MyCalculation(LedgerOp):
    def __init__(self, data_adapter: Data):
        self.data_adapter = data_adapter

    def run(self):
        self.data_adapter.latest().select("nA").execute().squeeze()


op = MyCalculation(data_adapter)
op.run()

There are various off-the-shelf records and operators.